# Handling Contractions and Alternative Words

As part of the scoring process, contractions were expanded and words with possible alternatives were replaced.

Before computing intelligibility scores, we expanded all contractions (e.g. `you're` → `you are`).
Additionally, we accounted for phonetically identical alternatives, including those arising from misspellings (e.g. `your` and `you're`)

For this purpose, we created a class that takes a `CSV` file containing word alternatives.
The class uses these alternatives, together with an input string sequence, to generate a list of possible string variations.

## Class AlternativeWords
The class `AlternativeWords` takes a CSV file (without a header) in its constructor and initializes a regex compiler.
The method ` accepts a string sequence and returns a list of all possible spelling alternatives.

```python
class AlternativeWords:
    """
    Class to handle alternative spellings. The class takes a CSV file 
    with two columns:
    
    - Column 1: word or phrase to be replaced
    - Column 2: alternative spelling or phrase
    
    The class can generate all possible sentence forms by replacing words/phrases
    with their alternatives.
    """

    def __init__(self, alternative_file: str):
        """Constructor
        
        Args:
            alternative_file (str): Path to the CSV file with alternative words.
        """
        self.alternative_dict = defaultdict(list)
        with open(alternative_file, "r") as f:
            for line in f:
                parts = [x.strip() for x in line.strip().split(",", 1)]
                if len(parts) == 1:
                    k, v = parts[0], ""
                else:
                    k, v = parts
                self.alternative_dict[k.lower()].append(v.lower())

        # Create regex pattern
        pattern = "|".join(
            rf"\b{re.escape(k)}\b" if "'" not in k else rf"(?<!\w){re.escape(k)}(?!\w)"
            for k in self.alternative_dict.keys()
        )

        self.contra_re = re.compile(f"({pattern})", re.IGNORECASE)

    def make_sentence_forms(self, sentence: str | list) -> list[str]:
        """ Generate all possible forms of a sentence by expanding using alternatives.

        Args:
            sentence (str or list): Input sentence or list of sentences.
        Returns:
            list: List of all possible sentence forms.
        """
        if isinstance(sentence, str):
            sentence = [sentence]

        APOST = r"['\u2019]"
        token_re = re.compile(
            rf"[a-z]+(?:{APOST}[a-z]+)*(?:{APOST})?|{APOST}[a-z]+|[^\w\s]",
            re.IGNORECASE,
        )

        sentence_forms = []
        for s in sentence:
            parts = token_re.findall(s.lower())

            # For each part, list all possible variants
            options = [
                self.alternative_dict[p] + [p] if p in self.alternative_dict else [p]
                for p in parts
            ]

            sentence_forms += [" ".join(s).strip() for s in product(*options)]
        return list(set(sentence_forms))
```

### Example for Contractions

In the following example, we expand the contractions in the sequence `I don't know if I'll go to the party` and generate all possible alternatives.

In [1]:
from alternative_words import AlternativeWords

# Contractiona object using ```contractions.csv``` file
contractions = AlternativeWords("../input_files/contractions.csv")
# The input sequence
input_sequence = "I don't know if I'll go to the party"
# generate alternatives by expanding contractions
alternative_forms = contractions.make_sentence_forms(input_sequence)

In [5]:
from IPython.display import display, Markdown

display(Markdown("**Original form:**"))
display(Markdown(f"- `{input_sequence}`"))

# Alternatives
alt_md = "**Possible forms:**\n"
alt_md += "\n".join(f"{i}. `{alt}`" for i, alt in enumerate(alternative_forms, 1))

display(Markdown(alt_md))


**Original form:**

- `I don't know if I'll go to the party`

**Possible forms:**
1. `i do not know if i will go to the party`
2. `i do not know if i'll go to the party`
3. `i don't know if i'll go to the party`
4. `i don't know if i will go to the party`

### Example for Alternative Word

In the same way, we can generate a list of alternative spellings based on homophones.
For example, let’s generate alternatives for the sequence: `your great, you're fool`.

By default, the system does not perform bidirectional transformations. Consider the following transformation specified in the CSV file:

|      |       |
|------|--------|
| your | you're |

The system will replace all occurrences of your with you're, but the reverse (`you're` → `your`) will not occur unless explicitly specified in the CSV file.

In [6]:
from alternative_words import AlternativeWords

# Alternative object using ```alternative_words.csv``` file
alternatives = AlternativeWords("../input_files/alternative_words.csv")
# The input sequence
input_sequence = "your great, you're fool"
# generate alternative spellings
alternative_forms = alternatives.make_sentence_forms(input_sequence)

In [7]:
from IPython.display import display, Markdown

display(Markdown("**Original form:**"))
display(Markdown(f"- `{input_sequence}`"))

# Alternatives
alt_md = "**Possible forms:**\n"
alt_md += "\n".join(f"{i}. `{alt}`" for i, alt in enumerate(alternative_forms, 1))

display(Markdown(alt_md))

**Original form:**

- `your great, you're fool`

**Possible forms:**
1. `you're great , you're fool`
2. `your great , you're fool`